In [ ]:
"""
NIFTY 50 STOCK PRICE PREDICTION - ACADEMIC PROJECT
===================================================
Goal: Predict if stock price will be higher 60 trading days (~3 months) in future
Method: Binary classification using technical indicators
Models: Logistic Regression, Random Forest, XGBoost

Author: [Your Name]
Date: February 2026
"""

# ============================================================================
# CELL 1: IMPORTS AND SETUP
# ============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn imports
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    confusion_matrix, roc_curve, precision_recall_curve
)
from sklearn.model_selection import cross_val_score

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("="*60)
print("NIFTY 50 STOCK PREDICTION - ACADEMIC PROJECT")
print("="*60)


Mounted at /content/drive
NIFTY 50 STOCK PREDICTION - ACADEMIC PROJECT


In [ ]:
# ============================================================================
# CELL 2: LOAD AND INSPECT RAW DATA
# ============================================================================
print("\n[STEP 1] Loading raw data...")
csv_path = "/content/drive/MyDrive/nifty50_historical_data.csv"
df_raw = pd.read_csv(csv_path, parse_dates=['Date'])
df_raw = df_raw.sort_values(['Ticker', 'Date']).reset_index(drop=True)

print(f"Total rows loaded: {len(df_raw):,}")
print(f"Date range: {df_raw['Date'].min()} to {df_raw['Date'].max()}")
print(f"Unique stocks: {df_raw['Ticker'].nunique()}")
print(f"\nColumns in raw data:\n{df_raw.columns.tolist()}")



[STEP 1] Loading raw data...
Total rows loaded: 287,310
Date range: 1999-01-01 00:00:00+05:30 to 2026-01-30 00:00:00+05:30
Unique stocks: 49

Columns in raw data:
['Date', 'Ticker', 'Company_Name', 'Sector', 'Open', 'High', 'Low', 'Close', 'Volume', 'Dividend', 'Stock_Split', 'Daily_Return', 'Volatility_20D', 'MA_50', 'MA_200', 'Market_Cap', 'PE_Ratio', 'Forward_PE', 'PEG_Ratio', 'Price_to_Book', 'Dividend_Yield', 'EPS', 'Beta', '52Week_High', '52Week_Low']


In [ ]:
#=============================================================================
# CELL 3: DATA CLEANING - KEEP ONLY OHLCV (NO LEAKAGE)
# ============================================================================
print("\n[STEP 2] Data cleaning - removing pre-calculated features...")

# Keep ONLY raw price/volume data to avoid leakage
keep_columns = ['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume']
df = df_raw[keep_columns].copy()

# Check for missing values
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
print(f"\nMissing values:\n{missing_pct[missing_pct > 0]}")

# Drop rows with missing OHLCV data
df = df.dropna(subset=['Open', 'High', 'Low', 'Close', 'Volume'])
print(f"Rows after removing missing values: {len(df):,}")



[STEP 2] Data cleaning - removing pre-calculated features...

Missing values:
Series([], dtype: float64)
Rows after removing missing values: 287,310


In [ ]:

# ============================================================================
# CELL 4: FEATURE ENGINEERING (FROM SCRATCH - NO LEAKAGE)
# ============================================================================
print("\n[STEP 3] Engineering technical features from scratch...")

def create_technical_features(data):
    """
    Create technical indicators from OHLC data.

    IMPORTANT: All features use ONLY past data (no future leakage)
    - Returns: % change from previous close
    - Moving averages: average of past N closes
    - Volatility: standard deviation of past returns
    - RSI: Relative Strength Index (momentum indicator)
    """
    df = data.copy()

    # Group by ticker to calculate features per stock
    for ticker in df['Ticker'].unique():
        mask = df['Ticker'] == ticker
        ticker_data = df.loc[mask, 'Close']

        # 1. PRICE-BASED FEATURES
        # Simple returns (% change from previous day)
        df.loc[mask, 'return_1d'] = ticker_data.pct_change(1)
        df.loc[mask, 'return_5d'] = ticker_data.pct_change(5)
        df.loc[mask, 'return_20d'] = ticker_data.pct_change(20)

        # 2. MOVING AVERAGES
        # Use min_periods=period to ensure we have enough data
        df.loc[mask, 'ma_10'] = ticker_data.rolling(window=10, min_periods=10).mean()
        df.loc[mask, 'ma_50'] = ticker_data.rolling(window=50, min_periods=50).mean()
        df.loc[mask, 'ma_200'] = ticker_data.rolling(window=200, min_periods=200).mean()

        # Price relative to moving averages
        df.loc[mask, 'price_to_ma50'] = ticker_data / df.loc[mask, 'ma_50']
        df.loc[mask, 'price_to_ma200'] = ticker_data / df.loc[mask, 'ma_200']

        # 3. VOLATILITY (risk measure)
        returns = ticker_data.pct_change()
        df.loc[mask, 'volatility_20d'] = returns.rolling(window=20, min_periods=20).std()
        df.loc[mask, 'volatility_60d'] = returns.rolling(window=60, min_periods=60).std()

        # 4. MOMENTUM INDICATORS
        # RSI (Relative Strength Index) - measures overbought/oversold
        delta = ticker_data.diff()
        gain = delta.where(delta > 0, 0).rolling(window=14, min_periods=14).mean()
        loss = -delta.where(delta < 0, 0).rolling(window=14, min_periods=14).mean()
        rs = gain / loss
        df.loc[mask, 'rsi_14'] = 100 - (100 / (1 + rs))

        # 5. VOLUME-BASED FEATURES
        volume_data = df.loc[mask, 'Volume']
        df.loc[mask, 'volume_ma_20'] = volume_data.rolling(window=20, min_periods=20).mean()
        df.loc[mask, 'volume_ratio'] = volume_data / df.loc[mask, 'volume_ma_20']

        # 6. PRICE RANGE FEATURES
        df.loc[mask, 'high_low_ratio'] = df.loc[mask, 'High'] / df.loc[mask, 'Low']
        df.loc[mask, 'close_to_high'] = df.loc[mask, 'Close'] / df.loc[mask, 'High']

    return df

# Apply feature engineering
df = create_technical_features(df)

# Show feature creation progress
features_created = [
    'return_1d', 'return_5d', 'return_20d',
    'ma_10', 'ma_50', 'ma_200',
    'price_to_ma50', 'price_to_ma200',
    'volatility_20d', 'volatility_60d',
    'rsi_14', 'volume_ma_20', 'volume_ratio',
    'high_low_ratio', 'close_to_high'
]
print(f"Created {len(features_created)} technical features")
print(f"Features: {features_created}")


[STEP 3] Engineering technical features from scratch...
Created 15 technical features
Features: ['return_1d', 'return_5d', 'return_20d', 'ma_10', 'ma_50', 'ma_200', 'price_to_ma50', 'price_to_ma200', 'volatility_20d', 'volatility_60d', 'rsi_14', 'volume_ma_20', 'volume_ratio', 'high_low_ratio', 'close_to_high']


In [ ]:
# ============================================================================
# CELL 5: CREATE TARGET VARIABLE (60-DAY FUTURE PRICE)
# ============================================================================
print("\n[STEP 4] Creating prediction target...")

PREDICTION_HORIZON = 60  # 60 trading days ≈ 3 months

# Create future price for each stock
df['future_close'] = df.groupby('Ticker')['Close'].shift(-PREDICTION_HORIZON)

# Binary target: 1 if price goes up, 0 if down
df['target'] = (df['future_close'] > df['Close']).astype(int)

# Remove rows where we can't calculate future price (last 60 days per stock)
df_clean = df.dropna(subset=['future_close']).copy()

print(f"Prediction horizon: {PREDICTION_HORIZON} trading days (~3 months)")
print(f"Rows with valid targets: {len(df_clean):,}")
print(f"\nTarget distribution:")
print(df_clean['target'].value_counts(normalize=True).round(3))



[STEP 4] Creating prediction target...
Prediction horizon: 60 trading days (~3 months)
Rows with valid targets: 284,370

Target distribution:
target
1    0.621
0    0.379
Name: proportion, dtype: float64


In [ ]:
# ============================================================================
# CELL 6: TIME-BASED TRAIN/VALIDATION/TEST SPLIT
# ============================================================================
print("\n[STEP 5] Creating time-based data splits...")

"""
IMPORTANT: For time-series data, we CANNOT use random splits!
We must respect temporal order:
- Train: oldest 70% of dates
- Validation: middle 15% of dates (for model tuning)
- Test: newest 15% of dates (final evaluation)
"""

# Calculate split dates
train_cutoff = df_clean['Date'].quantile(0.70)
val_cutoff = df_clean['Date'].quantile(0.85)

# Create splits
train_df = df_clean[df_clean['Date'] <= train_cutoff].copy()
val_df = df_clean[(df_clean['Date'] > train_cutoff) &
                   (df_clean['Date'] <= val_cutoff)].copy()
test_df = df_clean[df_clean['Date'] > val_cutoff].copy()

print(f"Train set: {len(train_df):,} rows ({train_df['Date'].min()} to {train_df['Date'].max()})")
print(f"Validation set: {len(val_df):,} rows ({val_df['Date'].min()} to {val_df['Date'].max()})")
print(f"Test set: {len(test_df):,} rows ({test_df['Date'].min()} to {test_df['Date'].max()})")

# Check target distribution in each split
print("\nTarget distribution by split:")
print(f"Train - Up: {train_df['target'].mean():.3f}")
print(f"Val   - Up: {val_df['target'].mean():.3f}")
print(f"Test  - Up: {test_df['target'].mean():.3f}")



[STEP 5] Creating time-based data splits...
Train set: 199,061 rows (1999-01-01 00:00:00+05:30 to 2018-10-16 00:00:00+05:30)
Validation set: 42,679 rows (2018-10-17 00:00:00+05:30 to 2022-04-28 00:00:00+05:30)
Test set: 42,630 rows (2022-04-29 00:00:00+05:30 to 2025-11-04 00:00:00+05:30)

Target distribution by split:
Train - Up: 0.614
Val   - Up: 0.618
Test  - Up: 0.653


In [ ]:
# ============================================================================
# CELL 7: PREPARE FEATURES AND REMOVE NaN
# ============================================================================
print("\n[STEP 6] Preparing final feature sets...")

# Define feature columns (exclude non-features)
feature_cols = [
    'return_1d', 'return_5d', 'return_20d',
    'ma_10', 'ma_50', 'ma_200',
    'price_to_ma50', 'price_to_ma200',
    'volatility_20d', 'volatility_60d',
    'rsi_14', 'volume_ratio',
    'high_low_ratio', 'close_to_high'
]

# Remove any remaining NaN (from rolling windows at start of each stock)
train_clean = train_df.dropna(subset=feature_cols).copy()
val_clean = val_df.dropna(subset=feature_cols).copy()
test_clean = test_df.dropna(subset=feature_cols).copy()

# Prepare X and y
X_train = train_clean[feature_cols]
y_train = train_clean['target']

X_val = val_clean[feature_cols]
y_val = val_clean['target']

X_test = test_clean[feature_cols]
y_test = test_clean['target']

print(f"Final dataset sizes:")
print(f"  Train: {len(X_train):,} samples")
print(f"  Validation: {len(X_val):,} samples")
print(f"  Test: {len(X_test):,} samples")
print(f"\nNumber of features: {len(feature_cols)}")



[STEP 6] Preparing final feature sets...
Final dataset sizes:
  Train: 187,343 samples
  Validation: 42,679 samples
  Test: 42,630 samples

Number of features: 14


In [ ]:

# ============================================================================
# CELL 8: MODEL 1 - LOGISTIC REGRESSION (BASELINE)
# ============================================================================
print("\n" + "="*60)
print("MODEL 1: LOGISTIC REGRESSION (BASELINE)")
print("="*60)

# Create pipeline with scaling and balanced class weights
logreg_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=1000,
        class_weight='balanced',  # Handle class imbalance
        random_state=42
    )
)

# Train model
print("\nTraining Logistic Regression...")
logreg_model.fit(X_train, y_train)

# Predictions
y_train_pred_lr = logreg_model.predict(X_train)
y_val_pred_lr = logreg_model.predict(X_val)
y_test_pred_lr = logreg_model.predict(X_test)

y_train_proba_lr = logreg_model.predict_proba(X_train)[:, 1]
y_val_proba_lr = logreg_model.predict_proba(X_val)[:, 1]
y_test_proba_lr = logreg_model.predict_proba(X_test)[:, 1]

# Evaluate on all sets
print("\n--- Performance Metrics ---")
print(f"Train Accuracy: {accuracy_score(y_train, y_train_pred_lr):.4f}")
print(f"Train ROC-AUC:  {roc_auc_score(y_train, y_train_proba_lr):.4f}")

print(f"\nValidation Accuracy: {accuracy_score(y_val, y_val_pred_lr):.4f}")
print(f"Validation ROC-AUC:  {roc_auc_score(y_val, y_val_proba_lr):.4f}")

print(f"\nTest Accuracy: {accuracy_score(y_test, y_test_pred_lr):.4f}")
print(f"Test ROC-AUC:  {roc_auc_score(y_test, y_test_proba_lr):.4f}")

print("\n--- Test Set Classification Report ---")
print(classification_report(y_test, y_test_pred_lr,
                          target_names=['Down (0)', 'Up (1)']))

print("\n--- Test Set Confusion Matrix ---")
cm_lr = confusion_matrix(y_test, y_test_pred_lr)
print(cm_lr)
print("\nInterpretation:")
print(f"  True Negatives (predicted down, was down): {cm_lr[0,0]}")
print(f"  False Positives (predicted up, was down): {cm_lr[0,1]}")
print(f"  False Negatives (predicted down, was up): {cm_lr[1,0]}")
print(f"  True Positives (predicted up, was up): {cm_lr[1,1]}")



MODEL 1: LOGISTIC REGRESSION (BASELINE)

Training Logistic Regression...

--- Performance Metrics ---
Train Accuracy: 0.5283
Train ROC-AUC:  0.5280

Validation Accuracy: 0.5007
Validation ROC-AUC:  0.5227

Test Accuracy: 0.4280
Test ROC-AUC:  0.4845

--- Test Set Classification Report ---
              precision    recall  f1-score   support

    Down (0)       0.34      0.68      0.45     14791
      Up (1)       0.63      0.29      0.40     27839

    accuracy                           0.43     42630
   macro avg       0.49      0.49      0.43     42630
weighted avg       0.53      0.43      0.42     42630


--- Test Set Confusion Matrix ---
[[10086  4705]
 [19680  8159]]

Interpretation:
  True Negatives (predicted down, was down): 10086
  False Positives (predicted up, was down): 4705
  False Negatives (predicted down, was up): 19680
  True Positives (predicted up, was up): 8159


In [ ]:
# ============================================================================
# CELL 9: MODEL 2 - RANDOM FOREST
# ============================================================================
print("\n" + "="*60)
print("MODEL 2: RANDOM FOREST")
print("="*60)

# Random Forest often works better for stock data (captures non-linear patterns)
rf_model = make_pipeline(
    StandardScaler(),
    RandomForestClassifier(
        n_estimators=100,  # 100 decision trees
        max_depth=10,      # Limit depth to prevent overfitting
        class_weight='balanced',
        random_state=42,
        n_jobs=-1  # Use all CPU cores
    )
)

print("\nTraining Random Forest...")
rf_model.fit(X_train, y_train)

# Predictions
y_val_pred_rf = rf_model.predict(X_val)
y_test_pred_rf = rf_model.predict(X_test)

y_val_proba_rf = rf_model.predict_proba(X_val)[:, 1]
y_test_proba_rf = rf_model.predict_proba(X_test)[:, 1]

# Evaluate
print("\n--- Performance Metrics ---")
print(f"Validation Accuracy: {accuracy_score(y_val, y_val_pred_rf):.4f}")
print(f"Validation ROC-AUC:  {roc_auc_score(y_val, y_val_proba_rf):.4f}")

print(f"\nTest Accuracy: {accuracy_score(y_test, y_test_pred_rf):.4f}")
print(f"Test ROC-AUC:  {roc_auc_score(y_test, y_test_proba_rf):.4f}")

print("\n--- Test Set Classification Report ---")
print(classification_report(y_test, y_test_pred_rf,
                          target_names=['Down (0)', 'Up (1)']))

cm_rf = confusion_matrix(y_test, y_test_pred_rf)
print("\n--- Test Set Confusion Matrix ---")
print(cm_rf)



MODEL 2: RANDOM FOREST

Training Random Forest...

--- Performance Metrics ---
Validation Accuracy: 0.4509
Validation ROC-AUC:  0.4717

Test Accuracy: 0.5077
Test ROC-AUC:  0.5254

--- Test Set Classification Report ---
              precision    recall  f1-score   support

    Down (0)       0.36      0.55      0.43     14791
      Up (1)       0.67      0.49      0.56     27839

    accuracy                           0.51     42630
   macro avg       0.52      0.52      0.50     42630
weighted avg       0.56      0.51      0.52     42630


--- Test Set Confusion Matrix ---
[[ 8076  6715]
 [14271 13568]]


In [ ]:
# ============================================================================
# CELL 10: FEATURE IMPORTANCE ANALYSIS (Random Forest)
# ============================================================================
print("\n" + "="*60)
print("FEATURE IMPORTANCE ANALYSIS")
print("="*60)

# Extract feature importances from Random Forest
rf_classifier = rf_model.named_steps['randomforestclassifier']
feature_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_classifier.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10).to_string(index=False))


FEATURE IMPORTANCE ANALYSIS

Top 10 Most Important Features:
       Feature  Importance
        ma_200    0.190492
         ma_50    0.147207
         ma_10    0.143618
volatility_60d    0.139768
price_to_ma200    0.102477
volatility_20d    0.078591
 price_to_ma50    0.057738
    return_20d    0.037055
        rsi_14    0.029148
  volume_ratio    0.018780


In [ ]:
# ============================================================================
# CELL 11: COMPREHENSIVE VISUALIZATIONS FOR ACADEMIC REPORT
# ============================================================================
print("\n[STEP 7] Generating visualizations for report...")

# Create a figure with multiple subplots
fig = plt.figure(figsize=(16, 12))

# 1. CONFUSION MATRICES COMPARISON
ax1 = plt.subplot(3, 3, 1)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=ax1)
ax1.set_title('Logistic Regression\nConfusion Matrix (Test Set)')
ax1.set_ylabel('Actual')
ax1.set_xlabel('Predicted')

ax2 = plt.subplot(3, 3, 2)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens', ax=ax2)
ax2.set_title('Random Forest\nConfusion Matrix (Test Set)')
ax2.set_ylabel('Actual')
ax2.set_xlabel('Predicted')

# 2. ROC CURVES
ax3 = plt.subplot(3, 3, 3)
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_test_proba_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_test_proba_rf)

ax3.plot(fpr_lr, tpr_lr, label=f'LogReg (AUC={roc_auc_score(y_test, y_test_proba_lr):.3f})', linewidth=2)
ax3.plot(fpr_rf, tpr_rf, label=f'RF (AUC={roc_auc_score(y_test, y_test_proba_rf):.3f})', linewidth=2)
ax3.plot([0, 1], [0, 1], 'k--', label='Random Guess')
ax3.set_xlabel('False Positive Rate')
ax3.set_ylabel('True Positive Rate')
ax3.set_title('ROC Curves (Test Set)')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 3. FEATURE IMPORTANCE
ax4 = plt.subplot(3, 3, 4)
top_features = feature_importance.head(10)
ax4.barh(range(len(top_features)), top_features['Importance'])
ax4.set_yticks(range(len(top_features)))
ax4.set_yticklabels(top_features['Feature'])
ax4.set_xlabel('Importance Score')
ax4.set_title('Top 10 Feature Importances\n(Random Forest)')
ax4.invert_yaxis()

# 4. PREDICTION DISTRIBUTION
ax5 = plt.subplot(3, 3, 5)
pred_dist = pd.DataFrame({
    'Actual': y_test.value_counts(),
    'LogReg': pd.Series(y_test_pred_lr).value_counts(),
    'RandomForest': pd.Series(y_test_pred_rf).value_counts()
})
pred_dist.plot(kind='bar', ax=ax5)
ax5.set_title('Prediction Distribution\n(Test Set)')
ax5.set_xlabel('Class (0=Down, 1=Up)')
ax5.set_ylabel('Count')
ax5.legend(['Actual', 'LogReg Pred', 'RF Pred'])
ax5.set_xticklabels(['Down', 'Up'], rotation=0)

# 5. PRECISION-RECALL CURVE
ax6 = plt.subplot(3, 3, 6)
prec_lr, rec_lr, _ = precision_recall_curve(y_test, y_test_proba_lr)
prec_rf, rec_rf, _ = precision_recall_curve(y_test, y_test_proba_rf)

ax6.plot(rec_lr, prec_lr, label='Logistic Regression', linewidth=2)
ax6.plot(rec_rf, prec_rf, label='Random Forest', linewidth=2)
ax6.set_xlabel('Recall')
ax6.set_ylabel('Precision')
ax6.set_title('Precision-Recall Curves')
ax6.legend()
ax6.grid(True, alpha=0.3)

# 6. TARGET DISTRIBUTION OVER TIME
ax7 = plt.subplot(3, 3, 7)
target_by_quarter = test_clean.groupby(test_clean['Date'].dt.to_period('Q'))['target'].mean()
ax7.plot(target_by_quarter.index.astype(str), target_by_quarter.values, marker='o')
ax7.set_xlabel('Quarter')
ax7.set_ylabel('Proportion of "Up" Cases')
ax7.set_title('Target Distribution Over Time\n(Test Set)')
ax7.grid(True, alpha=0.3)
plt.setp(ax7.xaxis.get_majorticklabels(), rotation=45)

# 7. MODEL COMPARISON METRICS
ax8 = plt.subplot(3, 3, 8)
metrics_comparison = pd.DataFrame({
    'Logistic Regression': [
        accuracy_score(y_test, y_test_pred_lr),
        roc_auc_score(y_test, y_test_proba_lr),
        precision_recall_curve(y_test, y_test_proba_lr)[0].mean(),
        precision_recall_curve(y_test, y_test_proba_lr)[1].mean()
    ],
    'Random Forest': [
        accuracy_score(y_test, y_test_pred_rf),
        roc_auc_score(y_test, y_test_proba_rf),
        precision_recall_curve(y_test, y_test_proba_rf)[0].mean(),
        precision_recall_curve(y_test, y_test_proba_rf)[1].mean()
    ]
}, index=['Accuracy', 'ROC-AUC', 'Avg Precision', 'Avg Recall'])

metrics_comparison.plot(kind='bar', ax=ax8)
ax8.set_title('Model Performance Comparison\n(Test Set)')
ax8.set_ylabel('Score')
ax8.set_xticklabels(metrics_comparison.index, rotation=45, ha='right')
ax8.legend()
ax8.grid(True, alpha=0.3, axis='y')

# 8. SAMPLE PREDICTIONS ANALYSIS
ax9 = plt.subplot(3, 3, 9)
# Show probability distribution for correct vs incorrect predictions
correct_probs = y_test_proba_rf[y_test == y_test_pred_rf]
incorrect_probs = y_test_proba_rf[y_test != y_test_pred_rf]

ax9.hist(correct_probs, bins=30, alpha=0.6, label='Correct', density=True)
ax9.hist(incorrect_probs, bins=30, alpha=0.6, label='Incorrect', density=True)
ax9.set_xlabel('Predicted Probability (Class 1)')
ax9.set_ylabel('Density')
ax9.set_title('Prediction Confidence\n(Random Forest)')
ax9.legend()

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/nifty50_ml_results.png', dpi=300, bbox_inches='tight')
print("\n✓ Saved comprehensive visualization to: /content/drive/MyDrive/nifty50_ml_results.png")
plt.show()


[STEP 7] Generating visualizations for report...


NameError: name 'plt' is not defined

In [ ]:
# ============================================================================
# CELL 12: SAVE RESULTS SUMMARY FOR REPORT
# ============================================================================
print("\n[STEP 8] Saving results summary...")

# Create summary report
summary_report = f"""
NIFTY 50 STOCK PRICE PREDICTION - RESULTS SUMMARY
================================================

PROJECT OVERVIEW:
- Dataset: Nifty 50 historical stock data
- Total samples: {len(df_clean):,}
- Prediction task: Binary classification (price up/down in 60 days)
- Time period: {df_clean['Date'].min()} to {df_clean['Date'].max()}

DATA SPLIT:
- Training set: {len(X_train):,} samples ({len(X_train)/len(df_clean)*100:.1f}%)
- Validation set: {len(X_val):,} samples ({len(X_val)/len(df_clean)*100:.1f}%)
- Test set: {len(X_test):,} samples ({len(X_test)/len(df_clean)*100:.1f}%)

FEATURES USED: {len(feature_cols)} technical indicators
{chr(10).join('- ' + f for f in feature_cols)}

MODEL 1: LOGISTIC REGRESSION
----------------------------
Test Set Performance:
- Accuracy: {accuracy_score(y_test, y_test_pred_lr):.4f}
- ROC-AUC: {roc_auc_score(y_test, y_test_proba_lr):.4f}
- Precision (Up): {classification_report(y_test, y_test_pred_lr, output_dict=True)['1']['precision']:.4f}
- Recall (Up): {classification_report(y_test, y_test_pred_lr, output_dict=True)['1']['recall']:.4f}

MODEL 2: RANDOM FOREST
---------------------
Test Set Performance:
- Accuracy: {accuracy_score(y_test, y_test_pred_rf):.4f}
- ROC-AUC: {roc_auc_score(y_test, y_test_proba_rf):.4f}
- Precision (Up): {classification_report(y_test, y_test_pred_rf, output_dict=True)['1']['precision']:.4f}
- Recall (Up): {classification_report(y_test, y_test_pred_rf, output_dict=True)['1']['recall']:.4f}

TOP 5 IMPORTANT FEATURES:
{feature_importance.head(5).to_string(index=False)}

INTERPRETATION:
--------------
1. ROC-AUC Score Meaning:
   - 0.50 = Random guessing (coin flip)
   - 0.50-0.60 = Poor performance
   - 0.60-0.70 = Fair performance
   - 0.70-0.80 = Good performance
   - 0.80-0.90 = Excellent performance
   - 0.90-1.00 = Outstanding performance

   Your models achieved: {roc_auc_score(y_test, y_test_proba_rf):.4f} ({"Poor" if roc_auc_score(y_test, y_test_proba_rf) < 0.6 else "Fair" if roc_auc_score(y_test, y_test_proba_rf) < 0.7 else "Good"})

2. Why Stock Prediction is Hard:
   - Markets are partially efficient (prices reflect available info)
   - Many factors beyond technical indicators (news, sentiment, macro events)
   - 60-day horizon is long-term (harder than short-term)
   - Class imbalance in bull/bear markets

3. Academic Insights:
   - Balanced class weights helped prevent "always predict up" problem
   - Random Forest performed {"better" if roc_auc_score(y_test, y_test_proba_rf) > roc_auc_score(y_test, y_test_proba_lr) else "similarly"} than Logistic Regression
   - Feature importance shows which technical indicators matter most
   - Time-based split ensures realistic evaluation

Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
"""

# Save summary
with open('/content/drive/MyDrive/nifty50_ml_summary.txt', 'w') as f:
    f.write(summary_report)

print("\n✓ Saved results summary to: /content/drive/MyDrive/nifty50_ml_summary.txt")
print("\n" + "="*60)
print("PROJECT COMPLETE!")
print("="*60)
print("\nFiles saved:")
print("1. /content/drive/MyDrive/nifty50_ml_results.png (visualizations)")
print("2. /content/drive/MyDrive/nifty50_ml_summary.txt (text report)")
print("\nThese files are ready to include in your academic report!")


[STEP 8] Saving results summary...

✓ Saved results summary to: /content/drive/MyDrive/nifty50_ml_summary.txt

PROJECT COMPLETE!

Files saved:
1. /content/drive/MyDrive/nifty50_ml_results.png (visualizations)
2. /content/drive/MyDrive/nifty50_ml_summary.txt (text report)

These files are ready to include in your academic report!


In [ ]:
# ============================================================================
# CELL 13: GENERATE UI-READY DATA FILES
# ============================================================================
"""
This cell extracts predictions and creates data files for the web UI.
Run this AFTER training both models (after Cell 12).

Generates:
1. predictions_detailed.csv - Every prediction with outcomes
2. stock_metadata.json - Company info per stock
3. stock_timeseries.parquet - Price history for charts
4. performance_summary.json - Overall metrics
"""

print("\n" + "="*70)
print("GENERATING UI-READY DATA FILES")
print("="*70)

import json
from datetime import datetime

# ============================================================================
# STEP 1: Extract predictions from test set with all details
# ============================================================================
print("\n[1/4] Extracting detailed predictions...")

# Get test set predictions (we'll use Random Forest as it performed better)
test_predictions = pd.DataFrame({
    'date': test_clean['Date'].values,
    'ticker': test_clean['Ticker'].values,
    'close_price': test_clean['Close'].values,
    'future_close': test_clean['future_close'].values,
    'actual_outcome': test_clean['target'].values,  # 1 = went up, 0 = went down
    'predicted_prob_up': y_test_proba_rf,  # Probability of going UP
    'predicted_class': y_test_pred_rf,  # 0 or 1
    'correct_prediction': (y_test_pred_rf == test_clean['target'].values).astype(int)
})

# Add actual return (for display)
test_predictions['actual_return_pct'] = (
    (test_predictions['future_close'] - test_predictions['close_price']) /
    test_predictions['close_price'] * 100
)

# Add prediction details
test_predictions['prediction_date'] = test_predictions['date']
test_predictions['target_date'] = test_predictions['date'] + pd.Timedelta(days=60)
test_predictions['horizon_days'] = 60

# Add confidence level (categorical)
def get_confidence(prob):
    if prob < 0.4:
        return 'High Bearish'
    elif prob < 0.5:
        return 'Low Bearish'
    elif prob < 0.6:
        return 'Low Bullish'
    else:
        return 'High Bullish'

test_predictions['confidence_label'] = test_predictions['predicted_prob_up'].apply(get_confidence)

# Add key technical indicators for this prediction
# (We'll add the most important features)
test_predictions['ma_50'] = test_clean['ma_50'].values
test_predictions['ma_200'] = test_clean['ma_200'].values
test_predictions['rsi_14'] = test_clean['rsi_14'].values
test_predictions['volatility_20d'] = test_clean['volatility_20d'].values
test_predictions['return_20d'] = test_clean['return_20d'].values

print(f"✓ Extracted {len(test_predictions):,} predictions from test set")
print(f"  Date range: {test_predictions['date'].min()} to {test_predictions['date'].max()}")

# ============================================================================
# STEP 2: Create stock metadata (company info, win rates per stock)
# ============================================================================
print("\n[2/4] Creating stock metadata...")

# Get unique company names and sectors from original dataset
company_info = df_raw[['Ticker', 'Company_Name', 'Sector']].drop_duplicates()

stock_metadata = {}

for ticker in test_predictions['ticker'].unique():
    ticker_data = test_predictions[test_predictions['ticker'] == ticker]
    company_row = company_info[company_info['Ticker'] == ticker].iloc[0]

    # Calculate stats for this stock
    total_preds = len(ticker_data)
    correct_preds = ticker_data['correct_prediction'].sum()
    win_rate = correct_preds / total_preds if total_preds > 0 else 0

    # Calculate average returns
    wins = ticker_data[ticker_data['actual_outcome'] == 1]
    losses = ticker_data[ticker_data['actual_outcome'] == 0]

    avg_return_up = wins['actual_return_pct'].mean() if len(wins) > 0 else 0
    avg_return_down = losses['actual_return_pct'].mean() if len(losses) > 0 else 0

    # Latest data point
    latest = ticker_data.sort_values('date').iloc[-1]

    stock_metadata[ticker] = {
        'company_name': company_row['Company_Name'],
        'sector': company_row['Sector'],
        'total_predictions': int(total_preds),
        'correct_predictions': int(correct_preds),
        'model_win_rate': round(float(win_rate), 4),
        'actual_up_percentage': round(float((ticker_data['actual_outcome'] == 1).mean()), 4),
        'avg_return_when_up': round(float(avg_return_up), 2),
        'avg_return_when_down': round(float(avg_return_down), 2),
        'latest_prediction_date': str(latest['date'].date()),
        'latest_close_price': round(float(latest['close_price']), 2),
        'latest_prediction_prob': round(float(latest['predicted_prob_up']), 4)
    }

print(f"✓ Created metadata for {len(stock_metadata)} stocks")

# ============================================================================
# STEP 3: Create time series data for charts
# ============================================================================
print("\n[3/4] Creating time series data for charts...")

# Get historical OHLCV + technical indicators for all stocks
# Use the cleaned dataset with features
timeseries_data = df_clean[
    ['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume',
     'ma_10', 'ma_50', 'ma_200', 'rsi_14', 'volatility_20d',
     'return_1d', 'return_5d', 'return_20d']
].copy()

# Rename for clarity
timeseries_data.columns = [
    'date', 'ticker', 'open', 'high', 'low', 'close', 'volume',
    'ma_10', 'ma_50', 'ma_200', 'rsi_14', 'volatility_20d',
    'return_1d', 'return_5d', 'return_20d'
]

print(f"✓ Created time series with {len(timeseries_data):,} rows")
print(f"  Columns: {len(timeseries_data.columns)} (OHLCV + indicators)")

# ============================================================================
# STEP 4: Create performance summary
# ============================================================================
print("\n[4/4] Creating performance summary...")

performance_summary = {
    'project_info': {
        'title': 'Nifty 50 Stock Price Direction Prediction',
        'prediction_horizon_days': 60,
        'features_count': len(feature_cols),
        'model_type': 'Random Forest',
        'generated_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    },

    'data_summary': {
        'total_stocks': df_clean['Ticker'].nunique(),
        'total_data_points': len(df_clean),
        'date_range_start': str(df_clean['Date'].min().date()),
        'date_range_end': str(df_clean['Date'].max().date()),
        'train_size': len(X_train),
        'validation_size': len(X_val),
        'test_size': len(X_test)
    },

    'overall_performance': {
        'test_roc_auc': round(float(roc_auc_score(y_test, y_test_proba_rf)), 4),
        'test_accuracy': round(float(accuracy_score(y_test, y_test_pred_rf)), 4),
        'total_predictions': int(len(test_predictions)),
        'correct_predictions': int(test_predictions['correct_prediction'].sum()),
        'win_rate': round(float(test_predictions['correct_prediction'].mean()), 4)
    },

    'class_distribution': {
        'test_pct_up': round(float(y_test.mean()), 4),
        'test_pct_down': round(float(1 - y_test.mean()), 4)
    },

    'prediction_distribution': {
        'avg_predicted_prob_up': round(float(test_predictions['predicted_prob_up'].mean()), 4),
        'median_predicted_prob_up': round(float(test_predictions['predicted_prob_up'].median()), 4),
        'high_confidence_bullish': int((test_predictions['predicted_prob_up'] >= 0.6).sum()),
        'high_confidence_bearish': int((test_predictions['predicted_prob_up'] <= 0.4).sum()),
        'uncertain': int(((test_predictions['predicted_prob_up'] > 0.4) &
                       (test_predictions['predicted_prob_up'] < 0.6)).sum())
    },

    'by_sector': {},

    'top_features': dict(zip(
        feature_importance['Feature'].head(10).tolist(),
        feature_importance['Importance'].head(10).round(4).tolist()
    )),

    'model_comparison': {
        'logistic_regression_roc_auc': round(float(roc_auc_score(y_test, y_test_proba_lr)), 4),
        'random_forest_roc_auc': round(float(roc_auc_score(y_test, y_test_proba_rf)), 4)
    }
}

# Add sector-level performance
for sector in df_raw['Sector'].unique():
    sector_tickers = df_raw[df_raw['Sector'] == sector]['Ticker'].unique()
    sector_preds = test_predictions[test_predictions['ticker'].isin(sector_tickers)]

    if len(sector_preds) > 0:
        performance_summary['by_sector'][sector] = {
            'stocks_count': len(sector_tickers),
            'predictions_count': int(len(sector_preds)),
            'win_rate': round(float(sector_preds['correct_prediction'].mean()), 4),
            'avg_predicted_prob_up': round(float(sector_preds['predicted_prob_up'].mean()), 4)
        }

print(f"✓ Created performance summary")

# ============================================================================
# SAVE ALL FILES
# ============================================================================
print("\n" + "="*70)
print("SAVING FILES...")
print("="*70)

output_dir = "/content/drive/MyDrive/ui_data"

# Create directory if it doesn't exist
import os
os.makedirs(output_dir, exist_ok=True)

# 1. Save detailed predictions (CSV for easy inspection)
predictions_path = f"{output_dir}/predictions_detailed.csv"
test_predictions.to_csv(predictions_path, index=False)
print(f"\n✓ Saved: {predictions_path}")
print(f"  Size: {len(test_predictions):,} rows x {len(test_predictions.columns)} columns")

# 2. Save stock metadata (JSON)
metadata_path = f"{output_dir}/stock_metadata.json"
with open(metadata_path, 'w') as f:
    json.dump(stock_metadata, f, indent=2)
print(f"\n✓ Saved: {metadata_path}")
print(f"  Stocks: {len(stock_metadata)}")

# 3. Save time series (Parquet for efficiency)
timeseries_path = f"{output_dir}/stock_timeseries.parquet"
timeseries_data.to_parquet(timeseries_path, index=False)
print(f"\n✓ Saved: {timeseries_path}")
print(f"  Size: {len(timeseries_data):,} rows")

# 4. Save performance summary (JSON)
summary_path = f"{output_dir}/performance_summary.json"
with open(summary_path, 'w') as f:
    json.dump(performance_summary, f, indent=2)
print(f"\n✓ Saved: {summary_path}")

# ============================================================================
# VERIFICATION & PREVIEW
# ============================================================================
print("\n" + "="*70)
print("VERIFICATION & PREVIEW")
print("="*70)

# Show sample prediction
print("\n📊 Sample Prediction (Random):")
sample = test_predictions.sample(1).iloc[0]
print(f"  Ticker: {sample['ticker']}")
print(f"  Date: {sample['date'].date()}")
print(f"  Close Price: ₹{sample['close_price']:.2f}")
print(f"  Predicted Prob UP: {sample['predicted_prob_up']:.2%}")
print(f"  Predicted: {'UP' if sample['predicted_class'] == 1 else 'DOWN'}")
print(f"  Actual: {'UP ✓' if sample['actual_outcome'] == 1 else 'DOWN ✗'}")
print(f"  Actual Return: {sample['actual_return_pct']:+.2f}%")
print(f"  Correct: {'YES ✓' if sample['correct_prediction'] == 1 else 'NO ✗'}")

# Show file sizes
print("\n💾 Generated Files:")
for filename in ['predictions_detailed.csv', 'stock_metadata.json',
                 'stock_timeseries.parquet', 'performance_summary.json']:
    filepath = f"{output_dir}/{filename}"
    if os.path.exists(filepath):
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f"  {filename}: {size_mb:.2f} MB")

# Show top/bottom performing stocks
print("\n🏆 Top 5 Stocks by Model Win Rate:")
top_stocks = sorted(stock_metadata.items(),
                   key=lambda x: x[1]['model_win_rate'],
                   reverse=True)[:5]
for ticker, data in top_stocks:
    print(f"  {ticker}: {data['model_win_rate']:.1%} ({data['company_name']})")

print("\n📉 Bottom 5 Stocks by Model Win Rate:")
bottom_stocks = sorted(stock_metadata.items(),
                      key=lambda x: x[1]['model_win_rate'])[:5]
for ticker, data in bottom_stocks:
    print(f"  {ticker}: {data['model_win_rate']:.1%} ({data['company_name']})")

# Sector performance
print("\n📊 Performance by Sector:")
sector_perf = sorted(performance_summary['by_sector'].items(),
                    key=lambda x: x[1]['win_rate'],
                    reverse=True)
for sector, stats in sector_perf:
    print(f"  {sector}: {stats['win_rate']:.1%} win rate ({stats['predictions_count']} predictions)")

print("\n" + "="*70)
print("✅ DATA GENERATION COMPLETE!")
print("="*70)
print(f"\nAll files saved to: {output_dir}/")
print("\nYou can now use these files to build the web UI:")
print("  1. predictions_detailed.csv - For showing individual predictions")
print("  2. stock_metadata.json - For stock list/search")
print("  3. stock_timeseries.parquet - For charts")
print("  4. performance_summary.json - For dashboard metrics")


GENERATING UI-READY DATA FILES

[1/4] Extracting detailed predictions...
✓ Extracted 42,630 predictions from test set
  Date range: 2022-04-28 18:30:00 to 2025-11-03 18:30:00

[2/4] Creating stock metadata...
✓ Created metadata for 49 stocks

[3/4] Creating time series data for charts...
✓ Created time series with 284,370 rows
  Columns: 15 (OHLCV + indicators)

[4/4] Creating performance summary...
✓ Created performance summary

SAVING FILES...

✓ Saved: /content/drive/MyDrive/ui_data/predictions_detailed.csv
  Size: 42,630 rows x 18 columns

✓ Saved: /content/drive/MyDrive/ui_data/stock_metadata.json
  Stocks: 49

✓ Saved: /content/drive/MyDrive/ui_data/stock_timeseries.parquet
  Size: 284,370 rows

✓ Saved: /content/drive/MyDrive/ui_data/performance_summary.json

VERIFICATION & PREVIEW

📊 Sample Prediction (Random):
  Ticker: DRREDDY.NS
  Date: 2024-04-04
  Close Price: ₹1220.84
  Predicted Prob UP: 46.86%
  Predicted: DOWN
  Actual: UP ✓
  Actual Return: +5.51%
  Correct: NO ✗

💾 